# Model03

## Feature selection

In [35]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import joblib

import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV)
from sklearn.feature_selection import VarianceThreshold

from secomml.data_loader import load_secom

In [4]:
secom = (
    load_secom("../data/raw/secom.zip")
    .drop(columns=['timestamp'])
)
X = secom.drop(columns='faulty')
y = secom['faulty']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=1)
X_train

,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x580,x581,x582,x583,x584,x585,x586,x587,x588,x589
467,3044.88,2636.69,2230.7555,1281.7862,1.0038,100,111.5489,0.1235,1.4868,-0.0325,...,<NA>,<NA>,0.4987,0.0138,0.0036,2.7637,0.0193,0.0122,0.004,63.0838
401,2986.34,2318.95,2213.7556,1113.5599,0.7217,100,104.1667,0.1211,1.3967,0.0176,...,0.0046,233.2441,0.4962,0.0098,0.0031,1.9826,0.0061,0.0141,0.0046,233.2441
1358,3084.17,2451.85,2231.0555,1303.5386,0.9751,100,95.7878,0.1198,1.5337,-0.0281,...,<NA>,<NA>,0.496,0.0123,0.003,2.4887,0.0077,0.0149,0.0041,193.4633
1554,3031.78,2528.55,2182.5555,1261.0898,1.211,100,112.2922,0.1252,1.3485,0.0151,...,0.0098,117.0945,0.5025,0.0138,0.0039,2.7512,0.0235,0.0275,0.0098,117.0945
833,3021.83,2419.83,2205.7445,1363.1048,1.0518,100,101.8644,0.122,1.4108,-0.0046,...,<NA>,<NA>,0.4934,0.0123,0.004,2.4923,0.0545,0.0184,0.0055,33.7876
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
669,3057.31,2481.53,2214.9333,1663.7024,1.0203,100,100.4456,0.1247,1.483,-0.0328,...,<NA>,<NA>,0.5037,0.0175,0.0039,3.4736,0.0263,0.0117,0.0033,44.3686
821,3009.71,2565.53,2224.6778,1308.6479,1.3907,100,101.1333,0.1208,1.444,-0.0079,...,<NA>,<NA>,0.4979,0.0078,0.0024,1.5724,0.0275,0.0215,0.0076,78.1199
213,3034.34,2631.47,2179.0445,2028.2208,1.5552,100,95.4256,0.1234,1.4281,0.0049,...,<NA>,<NA>,0.4968,0.0128,0.0035,2.5829,0.0298,0.0116,0.0039,38.9781
1556,3025.21,2503.3,2179.7333,3085.3781,1.4843,100,82.2467,0.1248,1.3687,-0.007,...,0.0016,46.1076,0.5019,0.0158,0.0043,3.1428,0.012,0.0055,0.0016,46.1076


### Pipeline & CT

In [7]:
imp_varsel_scaler = make_pipeline(
    SimpleImputer(strategy='mean'),
    VarianceThreshold(threshold=0.0),
    StandardScaler()
)
features = X_train.columns
ct = make_column_transformer(
    (imp_varsel_scaler, features),
    remainder='passthrough'
)
logreg = LogisticRegression(solver='liblinear', max_iter=300, random_state=1)
pipe = make_pipeline(ct, logreg)

%time cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

CPU times: user 1.5 s, sys: 64.7 ms, total: 1.57 s
Wall time: 1.57 s


np.float64(0.8885106382978722)

### RFE

#### Test

In [11]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
rfe = RFE(
    estimator=LinearRegression(),
    n_features_to_select=10,
)
X_train_transformed = imp_varsel_scaler.fit_transform(X_train)
%time rfe.fit(X_train_transformed, y_train)
rfe.get_feature_names_out()

CPU times: user 5min 34s, sys: 261 ms, total: 5min 35s
Wall time: 22.5 s


array(['x32', 'x34', 'x133', 'x143', 'x163', 'x165', 'x232', 'x242',
       'x262', 'x264'], dtype=object)

In [12]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
rfe = RFE(
    estimator=LogisticRegression(),
    n_features_to_select=10,
)
X_train_transformed = imp_varsel_scaler.fit_transform(X_train)
%time rfe.fit(X_train_transformed, y_train)
rfe.get_feature_names_out()

CPU times: user 25min 30s, sys: 814 ms, total: 25min 31s
Wall time: 1min 45s


array(['x17', 'x198', 'x217', 'x239', 'x299', 'x345', 'x415', 'x450',
       'x452', 'x461'], dtype=object)

#### RFE in Pipeline

In [14]:
imp_varsel_scaler = make_pipeline(
    SimpleImputer(strategy='mean'),
    VarianceThreshold(threshold=0.0),
    StandardScaler(),
    RFE(
        estimator=LogisticRegression(),
        n_features_to_select=10,
    )
)
features = X_train.columns
ct = make_column_transformer(
    (imp_varsel_scaler, features),
    remainder='passthrough'
)
logreg = LogisticRegression(solver='liblinear', max_iter=300, random_state=1)
pipe = make_pipeline(ct, logreg)

%time cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

CPU times: user 5min 23s, sys: 186 ms, total: 5min 23s
Wall time: 33 s


np.float64(0.9302127659574466)

In [29]:
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[float64](2,)","[0.,1.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](590,)","['x0','x1','x2',...,'x587','x588','x589']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,590
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('pipeline', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining colum

#### GridSearch

In [17]:
list(pipe.get_params())

['memory',
 'steps',
 'transform_input',
 'verbose',
 'columntransformer',
 'logisticregression',
 'columntransformer__n_jobs',
 'columntransformer__remainder',
 'columntransformer__sparse_threshold',
 'columntransformer__transformer_weights',
 'columntransformer__transformers',
 'columntransformer__verbose',
 'columntransformer__verbose_feature_names_out',
 'columntransformer__pipeline',
 'columntransformer__pipeline__memory',
 'columntransformer__pipeline__steps',
 'columntransformer__pipeline__transform_input',
 'columntransformer__pipeline__verbose',
 'columntransformer__pipeline__simpleimputer',
 'columntransformer__pipeline__variancethreshold',
 'columntransformer__pipeline__standardscaler',
 'columntransformer__pipeline__rfe',
 'columntransformer__pipeline__simpleimputer__add_indicator',
 'columntransformer__pipeline__simpleimputer__copy',
 'columntransformer__pipeline__simpleimputer__fill_value',
 'columntransformer__pipeline__simpleimputer__keep_empty_features',
 'columntransf

In [32]:
np.linspace(1, 474//2, 7)

array([  1.        ,  40.33333333,  79.66666667, 119.        ,
       158.33333333, 197.66666667, 237.        ])

In [33]:
params = joblib.load('../output/params_model02.joblib')
params['columntransformer__pipeline__rfe__n_features_to_select'] = [1, 10, 40, 80, 120, 160, 200, 240]
params['columntransformer__pipeline__rfe__estimator__C'] = [0.1]
params['columntransformer__pipeline__rfe__estimator__l1_ratio'] = [1]
joblib.dump(params, '../output/params_model03.joblib')
params

{'logisticregression__l1_ratio': [0, 1],
 'logisticregression__C': [0.1, 1, 10],
 'logisticregression__class_weight': [None, 'balanced'],
 'columntransformer__pipeline__simpleimputer__strategy': ['mean', 'median'],
 'columntransformer__pipeline__simpleimputer__add_indicator': [False, True],
 'columntransformer__pipeline__variancethreshold__threshold': array([1.00000000e-05, 2.15443469e-04, 4.64158883e-03, 1.00000000e-01,
        2.15443469e+00, 4.64158883e+01, 1.00000000e+03]),
 'columntransformer__pipeline__rfe__n_features_to_select': [1,
  10,
  40,
  80,
  120,
  160,
  200,
  240],
 'columntransformer__pipeline__rfe__estimator__C': [0.1],
 'columntransformer__pipeline__rfe__estimator__l1_ratio': [1]}

In [36]:
imp_varsel_scaler = make_pipeline(
    SimpleImputer(strategy='mean'),
    VarianceThreshold(threshold=0.0),
    StandardScaler(),
    RFE(estimator=LogisticRegression(solver='liblinear'))
)
features = X_train.columns
ct = make_column_transformer(
    (imp_varsel_scaler, features),
    remainder='passthrough'
)
logreg = LogisticRegression(solver='liblinear', max_iter=300, random_state=1)
pipe = make_pipeline(ct, logreg)

#grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy', n_jobs=-1)
grid = RandomizedSearchCV(pipe, params, cv=5, scoring='accuracy', n_jobs=-1)
%time grid.fit(X_train, y_train)

/home/lpanek/pkb/30-39-life/39-edu/ds/secom-ml/.venv/lib/python3.14/site-packages/sklearn/feature_selection/_rfe.py:301: UserWarning: Found n_features_to_select=120 > n_features=70. There will be no feature selection and all features will be kept.
  warnings.warn(
/home/lpanek/pkb/30-39-life/39-edu/ds/secom-ml/.venv/lib/python3.14/site-packages/sklearn/feature_selection/_rfe.py:301: UserWarning: Found n_features_to_select=120 > n_features=70. There will be no feature selection and all features will be kept.
  warnings.warn(
/home/lpanek/pkb/30-39-life/39-edu/ds/secom-ml/.venv/lib/python3.14/site-packages/sklearn/feature_selection/_rfe.py:301: UserWarning: Found n_features_to_select=120 > n_features=71. There will be no feature selection and all features will be kept.
  warnings.warn(
/home/lpanek/pkb/30-39-life/39-edu/ds/secom-ml/.venv/lib/python3.14/site-packages/sklearn/feature_selection/_rfe.py:301: UserWarning: Found n_features_to_select=120 > n_features=72. There will be no featur

CPU times: user 26.5 s, sys: 1.4 s, total: 27.9 s
Wall time: 2min 48s


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'columntransformer__pipeline__rfe__estimator__C': [0.1], 'columntransformer__pip...fe__estimator__l1_ratio': [1], 'columntransformer__pip...e__n_features_to_select': [1, 10, ...], 'columntransformer__pip...eimputer__add_indicator': [False, True], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificsco

In [37]:
grid.best_params_, grid.best_score_

({'logisticregression__l1_ratio': 1,
  'logisticregression__class_weight': None,
  'logisticregression__C': 0.1,
  'columntransformer__pipeline__variancethreshold__threshold': np.float64(1e-05),
  'columntransformer__pipeline__simpleimputer__strategy': 'median',
  'columntransformer__pipeline__simpleimputer__add_indicator': False,
  'columntransformer__pipeline__rfe__n_features_to_select': 10,
  'columntransformer__pipeline__rfe__estimator__l1_ratio': 1,
  'columntransformer__pipeline__rfe__estimator__C': 0.1},
 np.float64(0.9302127659574468))

In [38]:
%time cross_val_score(grid.best_estimator_, X_train, y_train, cv=5, scoring='accuracy').mean()

CPU times: user 57.3 s, sys: 108 ms, total: 57.4 s
Wall time: 57.4 s


np.float64(0.9302127659574468)

In [ ]:
from sklearn.metrics import (accuracy_score)
y_pred = grid.best_estimator_.predict(X_test)
print(f'accuracy_score:{accuracy_score(y_test, y_pred)}')

accuracy_score:0.9336734693877551


93.3% is naive estimator score